# Experiment log file

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from datetime import datetime, time 
from pathlib import Path
import pandas as pd
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
import numpy as np

Set path to log file name

In [ ]:
LOG_PATH = Path("../data/20181011_182540.log")
SMH_PATH = Path("../data/smh/")

### Parse QDSpy log file
Extract information from log file, resulting in a list of stimulus presentations (`stims`, list of dictionaries) will timing information and stimulus parameters

In [ ]:
nLinesTotal = 0
nLinesData = 0
nLinesErr = 0
nErr = 0
isStimStarted = False
stims = []
nStims = 0

with open(LOG_PATH, "r", encoding="utf-8", errors="ignore") as fLog:
    for line in fLog:
        # Extract elements of each line
        nLinesTotal += 1
        sDateTime = line[:15]
        sInfoType = line[15:23].strip().upper()
        sMsg = line[23:len(line) -1].strip()

        # Convert time stamp into a datetime        
        dt = datetime.strptime(sDateTime, "%Y%m%d_%H%M%S")
        if nLinesTotal == 1:
            # First line; take as start time
            dt_log_start = dt
            dt_last_end = dt_log_start

        # Filter for relevant information
        if sInfoType not in ["DATA"]:
            # Ignore filed
            continue

        # Convert data line into dictionary 
        sMsg = sMsg.replace("'", "\"")
        sMsg = sMsg.replace("\\\\", "/")
        sMsg = sMsg.replace("(", "[")
        sMsg = sMsg.replace(")", "]")

        try:
            data = json.loads(sMsg)
        except json.JSONDecodeError as e:
            # JSON parsing failed
            print(f"ERROR: parsing line {nLinesTotal-1} failed:")
            print(f"'{sMsg}'")
            nLinesErr += 1
            data = None

        # Get stimulus start/stop pairs
        try:
            stimState = data["stimState"].upper()
        except KeyError:
            stimState = None        


        if stimState:
            # Data contains stimulus information
            if stimState == "STARTED":
                if isStimStarted:
                    print("ERROR: Two consecutive stimulus starts")
                    nErr += 1

                isStimStarted = True
                iLineLastStart = nLinesData
                dt_start = dt
                t_diff = (dt -dt_log_start).total_seconds()
                t_diff_last = (dt -dt_last_end).total_seconds()
                stimInfo = dict(
                    {"index": nStims, 
                     "stimFileName": Path(data["stimFileName"]).name,
                     "stimPath": str(Path(data["stimFileName"]).parent),
                     "stimMD5": data["stimMD5"],
                     "t_abs_s": t_diff,
                     "t_since_last_s": t_diff_last,
                     "t_start": dt.time()}

                )

            elif stimState in ["ABORTED", "FINISHED"]:
                if isStimStarted:
                    # Check if stimulus end belongs to stimulus start
                    fn = str(Path(stimInfo["stimPath"], stimInfo["stimFileName"])).replace("\\", "/")
                    if not data["stimFileName"] == fn:
                        print("ERROR: File paths for stimulus start and end differ")
                        nErr += 1

                    # Append stimulus list entry
                    dt_last_end = dt
                    t_diff = (dt -dt_start).total_seconds()
                    stimInfo.update(
                        {"aborted": stimState == "ABORTED",
                         "t_end": dt.time(),
                         "t_dur_s": t_diff}
                    )
                    stims.append(stimInfo)
                    nStims += 1
                    isStimStarted = False
                else:
                    print("ERROR: Stimulus end w/o start?")    
                    nErr += 1
        else:
            # Other information
            try:
                _ = data["nFrames"]
                isFrameInfo = True
            except KeyError:
                isFrameInfo = False

            if isFrameInfo:
                # Information about stimulus presentation statistics
                stims[nStims -1].update(
                    {"t_dur_s_calc": data["nFrames"] /data["avgFreq_Hz"],
                     "nDroppedFrames": data["nDroppedFrames"]}
                )
            else:            
                if nLinesData > iLineLastStart and nLinesData < iLineLastStart +3:
                    # Last start was only up to 2 lines before
                    stims[nStims -1].update(
                        {"params": data}
                    )
                else:
                    print("ERROR: Data w/o start??")    
                    nErr += 1

        '''
        print(f"line #{nLinesData}:")
        print(dt)
        print(data)
        '''
        nLinesData += 1

    print(f"{nLinesData} of {nLinesTotal} line(s) extracted.")
    print(f"{nLinesErr} line(s) failed parsing, {nErr} error(s) occurred post-processing.")    

Print stimulus presentation table

In [ ]:
print("  #    start   t [s] gap [s] dur [s] nFr/frq abort stimulus name")
print("--- -------- ------- ------- ------- ------- ----- ")

for stim in stims:
    s = f"{stim["index"]:3d} {stim["t_start"]} {stim["t_abs_s"]:7.0f} "
    s += f"{stim["t_since_last_s"]:7.0f} {stim["t_dur_s"]:7.0f} {stim["t_dur_s_calc"]:7.0f} "  
    s += f"{"  y   " if stim["aborted"] else "      "} "
    s += f"`{Path(stim["stimFileName"]).name}`"

    print(s)

Example for information contained in one stimulus presentation entry

In [ ]:
stims[1]

... and as a pandas DataFrame, which is then written as a `.csv`file

In [ ]:
df_stims = pd.DataFrame(stims)
df_stims.to_csv("stims.csv", sep=";", decimal=",")
df_stims

### Parse ScanM `.smh` header files 
... to extract recording informations

In [ ]:
def stamps2datetime(s_date :str, s_time :str) -> datetime:
    """ Convert date and time stamps from .SMH header files 
    """
    d = datetime.strptime(s_date, '%Y-%m-%d').date()
    h, m, s, *_ = s_time.split('-')    
    return datetime.combine(d, time(int(h), int(m), int(s)))

def sec_to_hms(sec):
    s = int(sec)
    h = s // 3600
    m = (s % 3600) // 60
    s2 = s % 60
    return f"{h:02d}:{m:02d}:{s2:02d}"

def time_to_seconds(t):
    return t.hour * 3600 + t.minute * 60 + t.second

In [ ]:
from scanm.scanm_smp import SMP

nFiles = 0
smh_files = list(SMH_PATH.glob("*.smh"))
scmf = SMP()
smhs = []

# Loop through files and read data
for smh in smh_files:
    print(f"{nFiles:4d}: Reading {smh.name} ...")

    err_load_smh = scmf.loadSMH(smh, verbose=False)
    nFiles += 1

    dt = stamps2datetime(scmf._kvPairDict["DateStamp"][2], scmf._kvPairDict["TimeStamp"][2])
    info = dict(
        {"fName": smh.name,
         "date": dt.date(),
         "time": dt.time(),
         "xyz_coord_um": [
           scmf._kvPairDict["XCoord_um"][2], 
           scmf._kvPairDict["YCoord_um"][2], 
           scmf._kvPairDict["ZCoord_um"][2]]}
    )
    smhs.append(info)
    nFiles += 1

... as pandas DataFrame, written into a `.csv` file

In [ ]:
df_smhs = pd.DataFrame(smhs)
df_smhs.to_csv("smhs.csv", sep=";", decimal=",")
df_smhs

Plot field positions from `.smh` files

In [ ]:
df = pd.DataFrame(smhs)

# Keep only rows with valid xyz_coord_um lists
df = df[df['xyz_coord_um'].notna()].copy()

# extract x, y (assumes [x,y,z] or similar)
df['x'] = df['xyz_coord_um'].apply(lambda v: float(v[0]) if v and len(v) >= 2 else np.nan)
df['y'] = df['xyz_coord_um'].apply(lambda v: float(v[1]) if v and len(v) >= 2 else np.nan)

df['t_seconds'] = df['time'].apply(time_to_seconds)

# drop invalid rows
df_plot = df.dropna(subset=['x', 'y', 't_seconds']).copy()

fig, ax = plt.subplots(figsize=(7,7))
sc = ax.scatter(df_plot['x'], df_plot['y'], c=df_plot['t_seconds'], cmap='viridis', s=50, edgecolor='k', lw=0.3)
cbar = fig.colorbar(sc, ax=ax)

# format colorbar ticks as HH:MM:SS
cbar.ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda val, pos: sec_to_hms(val)))
cbar.set_label('time (HH:MM:SS)')

ax.set_xlabel('X [um]')
ax.set_ylabel('Y [um]')
ax.set_title('SMH files: X vs Y (colored by time)')
ax.grid(True)

# enforce aspect ratio 1:1 and preserve it when resizing
ax.set_aspect(1.0)
ax.set_adjustable('box')

plt.show()

## Generate a map of the stimulus presentation


**(1) Area illuminated by the different stimuli**

Get illuminated area and mean illumination for the stimuli.   
Note that it would be better to get the parameters directly from `stims`, but here we just use the most important stimulus parameters for an estimate.


In [ ]:
import stim_outlines

# Approx. field of view of stimulus through objective lens
# ESTIMATED: CHECK!
#FOV_diam = 1500  # Nikon 16x objective (CF175 LWD×16/0.8W, DIC N2)
FOV_diam = 1000   # W Plan-Apochromat 20x/1,0 DIC M27, Zeiss

# Create shapes (=areas illuminated by the stimuli)
MovingBar = dict(
    {'barDx_um': 1000.0, 
     'barDy_um': 300.0, 
     'nTrials': 3, 
     'vel_umSec': 1000.0, 
     'tMoveDur_s': 4.0, 
     'DirList': [0, 180, 45, 225, 90, 270, 135, 315], 
     'barColor': (255, 255, 255), 
     'bkgColor': (0, 0, 0)}
)
lEdge = MovingBar["barDy_um"]
trajLen = MovingBar["tMoveDur_s"] *MovingBar["vel_umSec"] +MovingBar["barDx_um"]
mb = stim_outlines.movingBar(lEdge, trajLen, MovingBar["DirList"], FOV_diam=FOV_diam)

RGC_Chirp = dict(
    {'tSteadyON_s': 3.0, 
     'tSteadyOFF2_s': 2.0, 
     'tSteadyMID_s': 2.0, 
     'chirpMaxFreq_Hz': 8.0, 
     'tSteadyOFF_s': 3.0, 
     'ContrastFreq_Hz': 2.0, 
     'nTrials': 5, 
     'dxStim_um': 1000, 
     'IHalf': 127, 
     'chirpDur_s': 8.0, 
     'IFull': 254}
)
chirp = stim_outlines.spot(diam=RGC_Chirp["dxStim_um"], FOV_diam=FOV_diam)

MouseCam_Left = dict(
    {'movparams_Train': {'nFr': 16200, 'dyFr': 56, 'dxFr': 56}, 
     'movName_Train': '//Katrin//RGCs//train_images_right.jpg', 
     'nFrRepeats': 2, 
     'movparams_Test': {'nFr': 750, 'dyFr': 56, 'dxFr': 56}, 
     'FrameRateMovie': 30.0, 
     'movName_Test': '//Katrin//RGCs//test_images_rand_right.jpg', 
     'movScale': (12.5, 12.5), 
     'movAlpha': 255, 
     'movOrient': 0, 
     'nFrPerMarker': 3, 
     'IndexName': 'RandomSequences', 
     'nTrials': 1, 
     'durSnippet_s': 5.0}
)
dx = MouseCam_Left["movparams_Train"]["dxFr"] *MouseCam_Left["movScale"][0]
dy = MouseCam_Left["movparams_Train"]["dyFr"] *MouseCam_Left["movScale"][1]
mouseMovie = stim_outlines.box(dx, dy, FOV_diam=FOV_diam)


**(2) Get stimulus sequence and recording positions**

Load currated file with stimulus and recording data.

Bringing together the data from the `.smh` files and the stimulus log file automatically is currently tricky, as the data here was recorded on two PCs with not well synchronized clocks. This needs to be done (e.g., sync time to time server automatically) in future experiments. Also, it would be good to record the scan position in the QDSpy `.log` file to have the field positions also for focus scans (where no recording file is written).


In [ ]:
CONSOL_PATH = Path("../data/experiment-overview_consolidated.csv")

df = pd.read_csv(CONSOL_PATH, on_bad_lines='warn', sep=';')

# Create figure for plotting
fig, ax = plt.subplots(figsize=(10, 10))

# Get time range for colormap normalization (only for rows with position data)
df_with_pos = df[df['pos_xyz'].notna()].copy()
if len(df_with_pos) > 0:
    t_min = df_with_pos['t_abs_s'].min()
    t_max = df_with_pos['t_abs_s'].max()
else:
    t_min, t_max = 0, 1

# Create colormap (using 'brg' - blue-red-green)
norm = Normalize(vmin=t_min, vmax=t_max)
cmap = cm.brg

# The alpha value equivalent to 1 sec stimulus exposure
alpha_per_s = 0.001

for index, row in df.iterrows():
    # Get coordinates 
    if row["pos_xyz"] is not np.nan:
        s = row["pos_xyz"][1:-1].split(",")
        pos_xyz = [float(s[i]) for i in range(len(s))]
    else:
        pos_xyz = None    

    # Get other parameters
    fStimName = row["stimFileName"]
    fRecName = "" if row["dataFileName"] is np.nan else row["dataFileName"]
    t_abs_s = row["t_abs_s"]
    t_dur_s = row["t_dur_s"]
    '''
    print(f"{row["index"]:3d} {fStimName:32s} {fRecName:30s} {pos_xyz}")
    '''

    if pos_xyz:
        # Plot stimuli only for presentations w/ position data
        x0, y0 = pos_xyz[0], pos_xyz[1]
        
        # Get color based on time
        edge_color = cmap(norm(t_abs_s))
        
        if fStimName.upper() in ["DS"]:
            # Plot moving bar outline at the current position
            poly = stim_outlines.movingBar(lEdge, trajLen, MovingBar["DirList"], x0=x0, y0=y0, FOV_diam=FOV_diam)
            x, y = poly.exterior.xy

            # Adjusting exposure to presentation duration and mean intensity (estimate)
            # TODO: Better estimate
            expos = min(1, t_dur_s *alpha_per_s *0.2)
            
            # Fill first (lower z-order), outline on top (higher z-order)
            ax.fill(x, y, color='yellow', alpha=expos, zorder=1)
            ax.plot(x, y, color=edge_color, linewidth=1.5, zorder=5)

        elif fStimName in ["Chirp"]:
            # Plot chirp outline at the current position
            poly = stim_outlines.spot(diam=RGC_Chirp["dxStim_um"], x0=x0, y0=y0, FOV_diam=FOV_diam)
            x, y = poly.exterior.xy

            # Adjusting exposure to presentation duration and mean intensity (estimate)
            # TODO: Better estimate
            expos = min(1, t_dur_s *alpha_per_s *0.5)
            
            # Fill first (lower z-order), outline on top (higher z-order)
            ax.fill(x, y, color='yellow', alpha=expos, zorder=1)
            ax.plot(x, y, color=edge_color, linewidth=1.5, zorder=5)

        elif fStimName in ["MouseCam_Right"]:
            # Plot movie outline at the current position
            poly = stim_outlines.box(dx, dy, x0=x0, y0=y0, FOV_diam=FOV_diam)
            x, y = poly.exterior.xy

            # Adjusting exposure to presentation duration and mean intensity (estimate)
            # TODO: Better estimate
            expos = min(1, t_dur_s *alpha_per_s *0.5)
            
            # Fill first (lower z-order), outline on top (higher z-order)
            ax.fill(x, y, color='yellow', alpha=expos, zorder=1)
            ax.plot(x, y, color=edge_color, linewidth=1.5, zorder=5)

# Add scatter points for recording positions
df_valid = df[df['pos_xyz'].notna()].copy()
df_valid['x'] = df_valid['pos_xyz'].apply(lambda v: float(v[1:-1].split(',')[0]))
df_valid['y'] = df_valid['pos_xyz'].apply(lambda v: float(v[1:-1].split(',')[1]))
sc = ax.scatter(df_valid['x'], df_valid['y'], c=df_valid['t_abs_s'], cmap='brg', 
                s=20, marker='o', zorder=10, edgecolor='k', linewidth=0.5)

# Add colorbar with same height as the plot
cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Time (s from start)')

ax.set_xlabel('X [um]')
ax.set_ylabel('Y [um]')
ax.set_title('Stimulus Presentation Map (colored by time)')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()